# Clase 4 — Funcionamiento interno de LLMs: tokens, atención y ventana de contexto

## Pregunta central

> **¿Qué pasa dentro de un LLM cuando genera una respuesta?**

## Idea principal

Un LLM no "piensa" ni "entiende": **predice el siguiente token** una y otra vez. Para hacerlo bien, usa un mecanismo llamado **atención** que decide qué partes del texto son relevantes, y trabaja dentro de un límite llamado **ventana de contexto** que define cuánto texto puede "ver" a la vez.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Explicar que un LLM predice el siguiente token, no "piensa".
- Explicar qué es la atención y por qué es el corazón de un LLM.
- Explicar qué es la ventana de contexto y por qué limita lo que el modelo puede ver.
- Ajustar los parámetros de generación (temperatura, top-k, max_tokens).
- Armar una integración completa: audio → texto → datos → respuesta.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | Tokens y vocabulario: lo que el modelo realmente ve |
| 2 | Predecir el siguiente token: logits, softmax y sampling |
| 3 | La atención: qué partes del texto importan |
| 4 | La ventana de contexto: el presupuesto de tokens |
| 5 | Parámetros de generación: temperatura, top-k, max_tokens |
| 6 | Actividad: integración audio → texto → datos → respuesta |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de modificar código, observá y describí el resultado.
3. Cambiá solamente las variables marcadas con `TODO`.
4. No es necesario implementar algoritmos desde cero.
5. Si aparece un término nuevo, buscá primero su definición en el glosario.

**Conexión con el programa:** la clase 2 nos dio el audio y la transcripción (Whisper); la clase 3 nos dio los datos estructurados. Esta clase explica qué pasa dentro del LLM que puede procesar todo eso, y la actividad integra el recorrido completo.


## Glosario mínimo

| Término | Explicación breve |
|---|---|
| Token | Pieza en la que se divide un texto (palabra, fragmento o signo) |
| Vocabulario | Conjunto de tokens que un modelo conoce |
| Logits | Puntajes crudos que el modelo da a cada token posible |
| Softmax | Regla que convierte puntajes en probabilidades |
| Greedy | Elegir siempre el token más probable |
| Sampling | Elegir el token con algo de azar |
| Atención | Mecanismo que decide qué partes del texto importan |
| Q, K, V | Consulta, clave y valor: los tres roles de la atención |
| Ventana de contexto | Cantidad máxima de tokens que el modelo puede ver |
| Temperatura | Parámetro que controla la creatividad |
| Top-k | Recortar la elección a los k tokens más probables |
| Top-p | Recortar la elección hasta cubrir una probabilidad acumulada |
| max_tokens | Límite de tokens que el modelo puede generar |
| ASR | Reconocimiento de voz: audio → texto (Whisper) |
| TTS | Síntesis de voz: texto → audio (MMS) |
| Embedding | Vector que representa el contenido de un texto |
| Entidad | Dato concreto dentro del texto (fecha, fármaco, dosis) |


In [ ]:
# --- Preparación: imports y configuración ---
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tiktoken

# Semilla fija: los números aleatorios salen iguales en cada ejecución.
SEED = 42
rng = np.random.default_rng(SEED)

# Tokenizador de ejemplo (el de los modelos GPT). Cada modelo tiene
# el suyo; tiktoken sirve para ver cómo funciona un tokenizador real.
tokenizador = tiktoken.get_encoding("cl100k_base")

print("Entorno listo.")


---
## 1. Tokens y vocabulario: lo que el modelo realmente ve

En la clase 3 vimos que un modelo no lee palabras: lee **tokens**. Cada token tiene un número (su **ID**) dentro del **vocabulario** del modelo.

> **Analogía del diccionario numerado.** Imaginá un diccionario donde cada palabra tiene un número. El modelo no ve la palabra "ibuprofeno": ve el número que le corresponde. Todo el texto que entra y sale del modelo es, en realidad, una secuencia de números.

```text
texto -> tokenizador -> IDs de tokens -> modelo
```

> **Pensalo así:** el tokenizador es la puerta de entrada y salida del modelo. Por un lado convierte texto en números; por el otro, convierte los números que genera el modelo de vuelta en texto.


In [ ]:
# --- Ver los IDs de los tokens ---
# Cada token tiene un número (ID) dentro del vocabulario del modelo.
FRASE = "El paciente toma ibuprofeno 600 mg cada 8 horas."

ids = tokenizador.encode(FRASE)
tokens = [tokenizador.decode([i]) for i in ids]

# Mostramos cada token con su ID.
tabla = pd.DataFrame({"token": tokens, "id": ids})
print("Frase:", FRASE)
print(f"\n{len(ids)} tokens con su ID:")
tabla


### Qué observar

- Cada token tiene un número único (ID) dentro del vocabulario.
- El modelo trabaja con esos números, no con las letras.
- El español suele necesitar más tokens que el inglés para el mismo contenido: eso tiene costo (lo veremos en la ventana de contexto).

> **Pregunta de interpretación:** ¿por qué creés que el modelo necesita convertir el texto en números antes de procesarlo?


---
## 2. Predecir el siguiente token: logits, softmax y sampling

Un LLM hace una sola cosa, una y otra vez: **predecir cuál es el siguiente token**. Cuando le damos "El paciente toma", el modelo calcula qué token tiene más sentido que siga.

> **Analogía del autocompletado.** Es como el autocompletado del celular, pero muchísimo más grande. El modelo mira lo que hay hasta ahora y propone la siguiente palabra. La diferencia es que el LLM hace esto con un vocabulario de decenas de miles de tokens y con muchísimo contexto.

### Los tres pasos de cada predicción

```text
tokens anteriores -> logits (puntajes) -> softmax (probabilidades) -> elegir token
```

1. **Logits:** el modelo da un puntaje crudo a cada token posible del vocabulario. Un puntaje alto = más probable.
2. **Softmax:** convierte esos puntajes en **probabilidades** (números entre 0 y 1 que suman 1).
3. **Elegir:** según la estrategia, se elige el token más probable (**greedy**) o se elige con algo de azar (**sampling**).

> **Pensalo así:** los logits son como las notas de un examen; el softmax las convierte en porcentajes; y la elección decide qué token se usa. La estrategia de elección (greedy o sampling) es lo que hace que un modelo suene más "seguro" o más "creativo".


In [ ]:
# --- Ver logits, softmax y la elección del token ---
# Simulamos los puntajes (logits) que el modelo da a 5 tokens posibles.
# En un modelo real estos puntajes salen de la red neuronal; aquí los
# inventamos para ver cómo funcionan los pasos.
TOKENS = ["turno", "cita", "medicamento", "dolor", "gracias"]
logits = np.array([3.2, 2.8, 1.5, 0.9, 0.2])

# Softmax: convierte los puntajes en probabilidades que suman 1.
def softmax(x):
    exp = np.exp(x - x.max())  # restamos el máximo para evitar números enormes
    return exp / exp.sum()

probabilidades = softmax(logits)

# Greedy: elige siempre el token más probable.
indice_greedy = probabilidades.argmax()

# Sampling: elige con azar, respetando las probabilidades.
indice_sampling = rng.choice(len(TOKENS), p=probabilidades)

tabla = pd.DataFrame({
    "token": TOKENS,
    "logit": logits,
    "probabilidad": probabilidades.round(3),
})
print("Probabilidades de cada token:")
tabla
print(f"\nGreedy elige: {TOKENS[indice_greedy]}")
print(f"Sampling eligió: {TOKENS[indice_sampling]}")


### Qué observar

- El softmax convierte puntajes en probabilidades que suman 1.
- **Greedy** siempre elige el mismo token (el más probable): es determinista.
- **Sampling** puede elegir un token distinto cada vez: es más variado.

> **Pregunta de interpretación:** si ejecutás la celda varias veces, ¿el sampling siempre elige lo mismo? ¿Por qué?

> **Pensalo así:** greedy es "siempre la opción más segura"; sampling es "a veces probar algo distinto". La **temperatura** (que veremos más adelante) controla qué tan "atrevido" es el sampling.


---
## 3. La atención: qué partes del texto importan

Cuando el modelo predice el siguiente token, no le da la misma importancia a todas las palabras anteriores. El mecanismo de **atención** decide **qué partes del texto importan más** para cada posición.

> **Analogía de la reunión.** Imaginá una reunión donde cada persona (token) debe resumir lo que se dijo. Para hacerlo, cada una "mira" a las demás y decide cuánto peso darle a cada una. La palabra "turno" le presta mucha atención a "dermatóloga" y "viernes", y poca a "hola". Eso es la atención: pesos que dicen "esto es lo importante".

### Los tres roles: Q, K y V

Para decidir esos pesos, cada token cumple tres roles:

| Rol | Qué hace | Analogía |
|---|---|---|
| Q (consulta) | "¿A quién le pregunto?" | La pregunta que hace el token |
| K (clave) | "¿Qué tengo para ofrecer?" | La etiqueta que muestra cada token |
| V (valor) | "¿Qué información doy?" | El contenido que aporta cada token |

El token con la consulta (Q) se compara con las claves (K) de los demás. Los que "encajan" reciben más peso. Luego se combinan los valores (V) según esos pesos.

> **Pensalo así:** Q es "lo que busco", K es "lo que ofrezco" y V es "lo que aporto". Si tu consulta encaja con mi clave, me prestás atención y usás mi valor. Así el modelo conecta palabras relacionadas aunque estén lejos en la frase.


In [ ]:
# --- Ver la atención con un ejemplo simplificado ---
# Simulamos la matriz de atención de una frase corta. Cada celda dice
# cuánta atención le presta un token a otro. En un modelo real estos
# pesos salen de la red; aquí los inventamos para ver el concepto.

FRASE_ATENCION = ["El", "paciente", "toma", "ibuprofeno", "600", "mg"]
# Matriz de atención (simplificada): filas = token que mira, columnas = token mirado.
# Los valores altos indican "le presto mucha atención".
atencion = np.array([
    [0.05, 0.10, 0.15, 0.30, 0.20, 0.20],  # El
    [0.05, 0.10, 0.20, 0.35, 0.15, 0.15],  # paciente
    [0.05, 0.10, 0.10, 0.40, 0.20, 0.15],  # toma
    [0.05, 0.10, 0.15, 0.30, 0.25, 0.15],  # ibuprofeno
    [0.05, 0.10, 0.15, 0.25, 0.30, 0.15],  # 600
    [0.05, 0.10, 0.15, 0.25, 0.20, 0.25],  # mg
])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(atencion, cmap="viridis", vmin=0, vmax=0.5)
ax.set_xticks(range(len(FRASE_ATENCION)))
ax.set_yticks(range(len(FRASE_ATENCION)))
ax.set_xticklabels(FRASE_ATENCION, rotation=45, ha="right")
ax.set_yticklabels(FRASE_ATENCION)
for i in range(len(FRASE_ATENCION)):
    for j in range(len(FRASE_ATENCION)):
        ax.text(j, i, f"{atencion[i, j]:.2f}", ha="center", va="center",
                color="white" if atencion[i, j] < 0.3 else "black", fontsize=9)
fig.colorbar(im, ax=ax, label="peso de atención")
ax.set_title("Matriz de atención (simplificada)")
plt.tight_layout()
plt.show()


### Qué observar

- Cada fila muestra a qué token le presta atención el token de esa fila.
- "toma" le presta mucha atención a "ibuprofeno" (el objeto de la acción).
- "600" y "mg" se prestan atención entre sí (forman una unidad: la dosis).

> **Pregunta de interpretación:** ¿qué token le presta más atención a "ibuprofeno"? ¿Tiene sentido?

> **Importante:** esta matriz es **simplificada** e inventada. En un modelo real hay muchas matrices de atención (una por "cabeza" y por capa), y los pesos salen de la red. Pero el concepto es este: el modelo conecta palabras relacionadas aunque estén lejos.


---
## 4. La ventana de contexto: el presupuesto de tokens

El modelo no puede "ver" todo el texto del mundo: solo puede ver una cantidad máxima de tokens a la vez. Ese límite se llama **ventana de contexto**.

> **Analogía del escritorio.** Imaginá un escritorio de tamaño fijo. Podés poner encima los papeles que quieras, pero cuando se llena, algo tiene que salir. La ventana de contexto es ese escritorio: define cuánto texto puede "tener a la vista" el modelo en cada momento.

### Por qué importa

- **Costo:** más tokens = más memoria y más tiempo.
- **Conversaciones:** cada mensaje del historial ocupa tokens. Una conversación larga puede llenar la ventana.
- **Contexto útil:** si el historial es muy largo, el modelo puede "olvidar" lo importante (porque lo sacó del escritorio).

> **Pensalo así:** la ventana de contexto es un **presupuesto**. Hay que decidir qué entra y qué sale. Por eso en las aplicaciones se recorta el historial o se resumen los mensajes viejos.


In [ ]:
# --- Calcular el presupuesto de tokens de una conversación ---
# Simulamos una conversación de un consultorio y contamos sus tokens.
# La ventana de contexto es el presupuesto: si la conversación lo supera,
# hay que recortar o resumir.

VENTANA_CONTEXTO = 512  # tokens disponibles (simplificado)

conversacion = [
    ("paciente", "Hola, quisiera pedir un turno con la dermatóloga para el viernes."),
    ("asistente", "Claro, ¿a qué hora le queda mejor? Tenemos mañana y tarde."),
    ("paciente", "Por la mañana, si es posible. ¿Tienen lugar con la doctora García?"),
    ("asistente", "Sí, el viernes a las 10. ¿Le confirmo el turno?"),
    ("paciente", "Perfecto, confirmado. ¿Me recuerdan que debo llevar algo?"),
]

# Contamos los tokens de cada mensaje y el total.
filas = []
total = 0
for rol, texto in conversacion:
    n = len(tokenizador.encode(texto))
    total += n
    filas.append({"rol": rol, "mensaje": texto[:40] + "...", "tokens": n})

tabla = pd.DataFrame(filas)
tabla.loc["TOTAL"] = ["", "", total]

print(f"Ventana de contexto: {VENTANA_CONTEXTO} tokens")
tabla
print(f"\nLa conversación usa {total} tokens de {VENTANA_CONTEXTO} disponibles.")
print(f"Quedan {VENTANA_CONTEXTO - total} tokens libres.")


### Qué observar

- Cada mensaje ocupa una cantidad de tokens.
- La conversación completa tiene un costo total en tokens.
- Si la conversación creciera, llegaría un punto donde no cabe en la ventana.

> **Pregunta de interpretación:** ¿qué pasaría si la conversación tuviera 20 mensajes? ¿Qué estrategia usarías para que quepa en la ventana?

> **Pensalo así:** en una aplicación real, el historial se gestiona: se recorta lo viejo, se resumen mensajes o se guarda solo lo importante. La ventana de contexto es la razón de ser de esas decisiones.


---
## 5. Parámetros de generación: temperatura, top-k, max_tokens

Cuando el modelo genera texto, hay varios parámetros que controlan **cómo** elige cada token. Son los "mandos" que ajustan el comportamiento del modelo.

| Parámetro | Qué controla | Valor bajo | Valor alto |
|---|---|---|---|
| Temperatura | La creatividad | Más determinista | Más variado |
| Top-k | Cuántos tokens considerar | Menos opciones | Más opciones |
| Top-p | Cubrir probabilidad acumulada | Más enfocado | Más abierto |
| max_tokens | Límite de la respuesta | Respuesta corta | Respuesta larga |

> **Analogía del chef.** La temperatura es como el picante: un poco da sabor, demasiado arruina el plato. Top-k es como elegir entre los mejores ingredientes. max_tokens es el tamaño del plato.

> **Pensalo así:** para tareas donde la respuesta debe ser **precisa** (como extraer datos), conviene temperatura baja. Para tareas **creativas** (como redactar un mensaje amable), una temperatura media puede ayudar. No hay un valor "correcto": depende de la tarea.


In [ ]:
# --- Ver cómo la temperatura cambia las probabilidades ---
# Con la misma lista de logits, la temperatura "aplana" o "acentúa"
# las probabilidades. Temperatura baja = más seguro; alta = más variado.

TOKENS = ["turno", "cita", "medicamento", "dolor", "gracias"]
logits = np.array([3.2, 2.8, 1.5, 0.9, 0.2])

def softmax_con_temperatura(x, temperatura):
    # Dividimos los logits por la temperatura antes del softmax.
    x = x / temperatura
    exp = np.exp(x - x.max())
    return exp / exp.sum()

# Probamos tres temperaturas distintas.
temperaturas = [0.2, 1.0, 2.0]
filas = []
for temp in temperaturas:
    probas = softmax_con_temperatura(logits, temp)
    for token, p in zip(TOKENS, probas):
        filas.append({"temperatura": temp, "token": token, "probabilidad": p.round(3)})

pd.DataFrame(filas).pivot(index="token", columns="temperatura", values="probabilidad")


### Qué observar

- Con **temperatura baja** (0.2), el token más probable ("turno") domina casi todo.
- Con **temperatura alta** (2.0), las probabilidades se reparten más parejo: hay más variedad.

> **Pregunta de interpretación:** ¿qué temperatura usarías para una tarea donde la respuesta debe ser exacta (como extraer un dato)? ¿Y para una tarea creativa?

> **Pensalo así:** la temperatura no cambia lo que el modelo "sabe", cambia cómo elige entre lo que sabe. Es un mando de comportamiento, no de conocimiento.


---
## 6. Actividad — integración: audio → texto → datos → respuesta

Ahora juntamos **todo lo que venimos viendo en el módulo 2**. El ejercicio recibe un **audio**, lo **transcribe** (ASR, clase 2), lo **clasifica y extrae datos** (clase 3) y genera una **respuesta** con un LLM (esta clase). El alumno puede alterar todos los parámetros.

> **Duración sugerida:** 30 minutos.

### El recorrido completo

```text
audio (clase 2)
    |
    v
ASR Whisper -> texto transcrito
    |
    v
clasificar + extraer entidades (clase 3)
    |
    v
LLM -> respuesta (esta clase)
    |
    v
TTS -> audio de respuesta (clase 2)
```

> **Pensalo así:** este es el esqueleto de un asistente conversacional de consultorio. Cada pieza la vimos por separado; ahora las conectamos. El alumno puede tocar los parámetros de cada etapa y ver cómo cambia el resultado final.

### Qué hacer

1. Ejecutá las celdas en orden para cargar los modelos (ASR, LLM, TTS).
2. Cambiá el **mensaje de entrada** (el texto que se convierte en audio).
3. Ajustá los **parámetros del LLM** (temperatura, max_tokens) y observá cómo cambia la respuesta.
4. Modificá las **anclas** de clasificación y los **diccionarios** de entidades.
5. Observá el **registro estructurado** final y la respuesta en audio.

> **Importante:** la primera ejecución descarga los modelos (Whisper Tiny, el LLM GGUF y MMS-TTS). Después quedan en caché. Si ya corriste las clases 2 y 3, ya están descargados.


In [ ]:
# ✏️ PASO 1: Cargar los modelos (ASR, LLM y TTS).
# La primera ejecución descarga los modelos y puede tardar unos minutos.
# Después quedan en caché.
from transformers import pipeline
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
import soundfile as sf
from IPython.display import Audio, display

# --- ASR: Whisper Tiny (audio -> texto) ---
asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=-1,
)

# --- LLM local (texto -> respuesta) ---
REPO_ID_LLM = "unsloth/LFM2.5-1.2B-Instruct-GGUF"
FILENAME_LLM = "LFM2.5-1.2B-Instruct-Q4_0.gguf"
ruta_modelo_llm = hf_hub_download(repo_id=REPO_ID_LLM, filename=FILENAME_LLM)
llm = Llama(model_path=ruta_modelo_llm, n_ctx=2048, n_gpu_layers=0, verbose=False)

# --- TTS: MMS español (texto -> audio) ---
tts = pipeline("text-to-speech", model="facebook/mms-tts-spa", device=-1)

print("Modelos listos: ASR (Whisper), LLM (LFM2.5), TTS (MMS).")


In [ ]:
# ✏️ PASO 2: Definir las herramientas de clasificación y extracción.
# TODO: modificá las anclas y los diccionarios para ajustar el comportamiento.

# --- Anclas de clasificación (clase 3) ---
ANCLAS = {
    "turno": "quiero pedir, cambiar o cancelar un turno o cita médica",
    "receta": "pregunta sobre un medicamento, dosis o receta",
    "consulta_medica": "descripción de un síntoma o dolor actual",
    "otro": "consulta administrativa sobre horarios, dirección o pagos",
}

# --- Diccionarios de entidades (clase 3) ---
MEDICAMENTOS = ["ibuprofeno", "paracetamol", "losartán", "metformina", "insulina"]
ESPECIALIDADES = ["dermatóloga", "cardióloga", "dentista", "clínica"]
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

# --- Encoder para la similitud (clase 3) ---
try:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2", device="cpu"
    )
    modo = "embeddings"
except Exception:
    encoder = None
    modo = "palabras compartidas (fallback)"

def similitud(a, b):
    if encoder is not None:
        va = encoder.encode([a], normalize_embeddings=True)[0]
        vb = encoder.encode([b], normalize_embeddings=True)[0]
        return float(va @ vb)
    pa = set(a.lower().split())
    pb = set(b.lower().split())
    return len(pa & pb) / max(1, len(pa | pb))

def clasificar(texto):
    resultados = [
        (categoria, similitud(texto, ancla))
        for categoria, ancla in ANCLAS.items()
    ]
    resultados.sort(key=lambda par: -par[1])
    return resultados[0]

def extraer_entidades(texto):
    texto_normalizado = texto.lower()
    entidades = {
        "medicamentos": [m for m in MEDICAMENTOS if m in texto_normalizado],
        "especialidades": [e for e in ESPECIALIDADES if e in texto_normalizado],
        "dias": [d for d in DIAS if d in texto_normalizado],
        "dosis": re.findall(r"\d+\s*(?:mg|ml)", texto_normalizado),
        "horas": re.findall(r"\d+\s*(?:hs|h)", texto_normalizado),
    }
    return {k: v for k, v in entidades.items() if v}

print(f"Herramientas listas (modo: {modo}).")


In [ ]:
# ✏️ PASO 3: Definir el mensaje de entrada y convertirlo en audio.
# TODO: cambiá este mensaje para probar distintos casos.
MENSAJE_ENTRADA = "Hola, me recetaron losartán 50 mg y quiero saber si lo tomo en ayunas."

# Convertimos el mensaje en audio con TTS (para simular la entrada de voz).
os.makedirs("salidas", exist_ok=True)
audio_entrada = tts(MENSAJE_ENTRADA)
ruta_entrada = os.path.join("salidas", "consulta_clase4.wav")
sf.write(ruta_entrada, audio_entrada["audio"], audio_entrada["sampling_rate"])

print("Mensaje de entrada:", MENSAJE_ENTRADA)
print("Audio guardado en:", ruta_entrada)
display(Audio(filename=ruta_entrada))


In [ ]:
# ✏️ PASO 4: Transcribir el audio con ASR (Whisper).
# El audio se convierte en texto. Este texto es la entrada del resto del pipeline.
transcripcion = asr(ruta_entrada, return_timestamps=True)["text"]

print("Audio transcrito por Whisper:")
print("-" * 50)
print(transcripcion)
print("-" * 50)


In [ ]:
# ✏️ PASO 5: Clasificar y extraer entidades de la transcripción.
# Usamos las herramientas de la clase 3 sobre el texto transcrito.
categoria, score = clasificar(transcripcion)
entidades = extraer_entidades(transcripcion)

print("Clasificación de la transcripción:")
print(f"  Categoría: {categoria}  (similitud {score:.2f})")
print(f"  Entidades: {entidades}")


In [ ]:
# ✏️ PASO 6: Generar la respuesta con el LLM.
# TODO: ajustá la temperatura y max_tokens para ver cómo cambia la respuesta.
TEMPERATURA = 0.2
MAX_TOKENS = 120

# El prompt le da al LLM la transcripción y los datos extraídos.
system_prompt = """
Sos el asistente de recepción de un consultorio.
Respondé de forma breve, amable y clara.
No diagnostiques, no indiques tratamientos, no inventes horarios ni confirmes acciones que no ejecutaste.
Si falta información, pedila. Si hay una consulta médica o una urgencia, indicá que se deriva a una persona.
"""

contexto = (
    f"Transcripción del paciente: {transcripcion}\n"
    f"Categoría detectada: {categoria}\n"
    f"Entidades detectadas: {entidades}\n"
    "Respondé al paciente:"
)

respuesta = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": contexto},
    ],
    temperature=TEMPERATURA,
    max_tokens=MAX_TOKENS,
)["choices"][0]["message"]["content"].strip()

print(f"Respuesta del LLM (temperatura={TEMPERATURA}, max_tokens={MAX_TOKENS}):")
print("-" * 50)
print(respuesta)
print("-" * 50)


In [ ]:
# ✏️ PASO 7: Convertir la respuesta en audio (TTS) y mostrar el registro.
# La respuesta se "lee" en voz alta, cerrando el ciclo voz -> texto -> voz.
audio_respuesta = tts(respuesta)
ruta_respuesta = os.path.join("salidas", "respuesta_clase4.wav")
sf.write(ruta_respuesta, audio_respuesta["audio"], audio_respuesta["sampling_rate"])

# Registro estructurado final: todo lo que la aplicación puede usar.
registro = {
    "mensaje_original": MENSAJE_ENTRADA,
    "transcripcion": transcripcion,
    "categoria": categoria,
    "similitud": round(score, 2),
    "entidades": entidades,
    "respuesta": respuesta,
    "requiere_revision_humana": bool(score < 0.5),
}

print("Registro estructurado final:")
for campo, valor in registro.items():
    print(f"  {campo}: {valor}")

print("\nRespuesta en audio:")
display(Audio(filename=ruta_respuesta))


### Para compartir al final

1. Mostrá el registro estructurado completo (audio → texto → datos → respuesta).
2. Indicá qué parámetro del LLM cambiaste y cómo cambió la respuesta.
3. Mostrá un caso donde la clasificación o la extracción falló y cómo lo corregiste.
4. Explicá en qué caso marcarías el mensaje para revisión humana.

> **Cierre:** este pipeline integra todo el módulo 2: el audio (clase 2), el lenguaje (clase 3) y el LLM (esta clase). Cada pieza es ajustable, y la decisión final siempre la toma una persona.


---

## Síntesis de la clase

- Un LLM predice el siguiente token, una y otra vez; no "piensa".
- Los logits se convierten en probabilidades con softmax; greedy y sampling eligen el token.
- La atención decide qué partes del texto importan, conectando palabras relacionadas.
- La ventana de contexto es el presupuesto de tokens; hay que gestionar qué entra y qué sale.
- Los parámetros (temperatura, top-k, max_tokens) controlan cómo elige el modelo.
- El pipeline audio → texto → datos → respuesta integra todo el módulo 2.

## Comprobación conceptual

Antes de continuar, intentá responder sin mirar el notebook:

1. ¿Cuál era el problema central de la clase?
2. ¿Qué entrada recibió el sistema y qué salida produjo?
3. ¿Qué decisión humana siguió siendo necesaria?
4. ¿Qué limitación observaste en el experimento?

Si podés explicarlo con tus propias palabras y justificarlo con un resultado visible, alcanzaste el objetivo introductorio.

## Puente con la próxima clase

La clase 5 profundiza el prompting avanzado y las salidas JSON. Ahora que entendés la ventana de contexto y los parámetros de generación, podés diseñar prompts que aprovechen mejor el modelo.

## Conexión con el track

Salud usará este pipeline para asistentes conversacionales: el audio se transcribe, se convierte en datos estructurados y el LLM genera una respuesta, siempre con validación humana en decisiones clínicas.

La implementación profunda, el trabajo con datasets reales y las decisiones de producción se desarrollarán en los módulos especializados.
